<p>
<font size='5' face='Georgia, Arial'>IIC-2233 Apunte Programación Avanzada</font><br>
<font size='1'>&copy; 2015 Karim Pichara - Christian Pieringer. Todos los derechos reservados. Editado por Equipo Docente IIC2233 2018-1 al 2026-2</font>
</p>

# Tabla de contenidos

1. [Ejecución de código de ejemplo del material que usa PyQt](#Ejecución-de-código-de-ejemplo-del-material-que-usa-PyQt)
2. [Ejemplo aplicado](#Ejemplo-aplicado)
    1. [Comparando los tres *back-ends*](#Comparando-los-tres-back-ends)
3. [Tips](#Tips)
    1. [`QMainWindow`](#QMainWindow)
    2. [`isAutoRepeat` en `keyPressEvent`](#isAutoRepeat-en-keyPressEvent)
    3. [Versiones de PyQt](#Versiones-de-PyQt)
4. [Errores frecuentes](#Errores-frecuentes)

## Ejecución de código de ejemplo del material que usa PyQt

Jupyter no fue creado para ejecutar interfaces gráficas de escritorio, por lo que en este _notebook_ el código se muestra como texto y **no** se ejecuta. Cada ejemplo está en la carpeta `scripts/`, y te recomendamos **fuertemente** ejecutarlo desde tu computador e interactuar con él.

**Importante:** al ejecutar un _script_, la terminal debe estar posicionada en la misma carpeta del archivo `.py`. Por ejemplo:

```bash
cd scripts/5-ejemplo-aplicado
python3 main.py
```

## Ejemplo aplicado

Para cerrar, juntemos todo lo de la semana: modularización, *front-end* y *back-end*, señales, *threads* y `QTimer`.

![](img/ejemplo_aplicado.gif)

En este ejemplo hay varios cuadrados que van de un extremo a otro de la ventana, cada uno a distinta velocidad. Al hacer *click* en un cuadrado, este cambia de color y se detiene. (Mini desafío: intenta detenerlos todos).

El ejemplo está en `scripts/5-ejemplo-aplicado/`, y lo interesante es que **el mismo juego está implementado con tres _back-ends_ distintos**, que se pueden intercambiar sin tocar el *front-end*. Los archivos son:

* `main.py`: el archivo principal a ejecutar. Crea el *front-end* y el *back-end*, y conecta sus señales.
* `parametros_general.py`: constantes que usan el *back-end* y el *front-end*.

**backend**
* `backend/logica_qthread.py`: cada cuadrado es un `QThread`. **Es el que se usa por defecto.**
* `backend/logica_thread.py`: cada cuadrado es un `threading.Thread`. Es idéntico al anterior, salvo la herencia y dos detalles: sus *threads* son *daemon*, y al cerrar se esperan con `join()` en vez de `wait()`.
* `backend/logica_qtimer.py`: cada cuadrado es un `QObject` con su propio `QTimer`.
* `backend/parametros_backend.py`: constantes que usa solo el *back-end*.

**frontend**
* `frontend/ventana.py`: la ventana y todos los elementos visuales.
* `frontend/parametros_frontend.py`: constantes que usa solo el *front-end*.

Para cambiar de *back-end*, basta con cambiar la línea 5 de `main.py`:

```python
from PyQt6.QtWidgets import QApplication
from frontend.ventana import Ventana
import sys
import parametros_general as p
from backend.logica_qthread import Juego

# El mismo juego está implementado con 3 back-ends distintos. Para probar
# otro, cambia la línea 5 por una de las siguientes:
# from backend.logica_thread import Juego   # threading.Thread
# from backend.logica_qtimer import Juego   # QTimer
```

### Comparando los tres *back-ends*

La diferencia entre ellos está en cómo se mueve cada cuadrado (`Icono`). Con `QThread` (y lo mismo con `Thread`), cada cuadrado tiene su propio ciclo con un `sleep` en `run`, y se detiene con una bandera:

```python
    def run(self) -> None:
        # Mientras pueda correr
        while self.correr:
            # Me muevo y luego espero un poco de tiempo.
            self.mover()

            # Descanso los MS indicados. sleep recibe la cantidad en segundos
            # por lo tanto, debemos dividir por 1000 para transformar p.TIEMPO_MOVIMIENTO
            # en segundos
            time.sleep(p.TIEMPO_MOVIMIENTO / 1000)
```

Con `QTimer`, en cambio, no hay ciclo ni `sleep`: el *timer* llama a `mover` cada `TIEMPO_MOVIMIENTO` milisegundos, y se detiene con `stop()`:

```python
        # Creamos nuestro QTimer encargado de ejecutar
        # "self.mover" cada p.TIEMPO_MOVIMIENTO
        self.timer_movimiento = QTimer(self)
        self.timer_movimiento.timeout.connect(self.mover)
        self.timer_movimiento.setInterval(p.TIEMPO_MOVIMIENTO)
```

En resumen:

| | `Thread` / `QThread` | `QTimer` |
|---|---|---|
| ¿Dónde se ejecuta `mover`? | En el *thread* de cada cuadrado | En el *thread* principal |
| ¿Cómo se detiene un cuadrado? | Bandera `correr = False` | `timer.stop()` |
| ¿Hay riesgo de concurrencia? | Sí, hay que cuidar el estado compartido | No |
| ¿Hay que detenerlo al cerrar? | Sí: bandera + `wait()`/`join()` | Basta con `stop()` |

Los tres funcionan, pero según la tabla del primer *notebook*, este es un caso de **tarea periódica y rápida**: mover un cuadrado unos pocos píxeles. Por eso `QTimer` es la opción más simple. Si además los cuadrados tuvieran que interactuar entre sí (por ejemplo, chocar), un solo `QTimer` para todo el juego, como en `2-qtimer/5_loop_de_juego.py`, sería todavía más cómodo.

Fíjate también en que `main.py` conecta `app.aboutToQuit` con el método `detener` del *back-end*, para que los *threads* o *timers* terminen ordenadamente al cerrar la aplicación (ver [Detener un QThread y cerrar limpio](3-qthread.ipynb#Detener-un-QThread-y-cerrar-limpio)).

## Tips

Esta sección reúne herramientas que no son el centro de la semana, pero que te serán útiles. Los ejemplos están en `scripts/5-recetario/`, y como usan archivos con *paths* relativos (imágenes), la terminal **debe** estar posicionada en esa carpeta:

```bash
cd scripts/5-recetario
python3 1_mainwindow.py
```

### `QMainWindow`

Las ventanas que hemos creado con `QWidget` son ventanas simples, donde ubicamos otros *widgets*. PyQt ofrece una ventana más completa, `QMainWindow`, que incluye el esqueleto clásico de una aplicación:

![](img/pyqt-mainwindow-layout.png)

- La **barra de menú** (`menuBar()`) agrupa los comandos de la aplicación en menús.
- La **barra de herramientas** (`addToolBar()`) da acceso rápido a los comandos más usados.
- La **barra de estado** (`statusBar()`) muestra información del estado de la aplicación. `statusBar()` es un método de `QMainWindow`.
- El ***central widget*** es el cuerpo de la ventana, y puede ser cualquier *widget*, incluso una de las ventanas de los ejemplos anteriores. Se define con `setCentralWidget(widget)`.

Este código se encuentra en el archivo `5-recetario/1_mainwindow.py`.

```python
import os
import sys

from PyQt6.QtCore import pyqtSignal
from PyQt6.QtGui import QIcon, QAction
from PyQt6.QtWidgets import QApplication, QMainWindow, QWidget
from PyQt6.QtWidgets import QPushButton, QLabel, QLineEdit

class MiVentana(QWidget):
    def __init__(self, status_bar_signal: pyqtSignal) -> None:
        super().__init__()
        self.status_bar_signal = status_bar_signal
        self.init_gui()

    def init_gui(self) -> None:
        """
        Este método inicializa el main widget y sus elementos.
        """
        self.label_text = QLabel("Texto", self)
        self.print_label = QLabel("Print texto:", self)
        self.line_edit = QLineEdit("", self)

        self.boton = QPushButton("&Procesar", self)
        self.boton.resize(self.boton.sizeHint())
        self.boton.clicked.connect(self.boton_callback)

        self.label_text.move(20, 15)
        self.line_edit.setGeometry(55, 15, 100, 20)
        self.print_label.move(20, 50)
        self.boton.move(20, 80)

    def boton_callback(self) -> None:
        """
        Este método es el encargado ejecutar una acción cada vez que el botón
        es presionado. En esta caso, realiza el cambio en print_label y el status bar
        mediate la emisión de una señal en la cual se envía el texto correspondiente.
        """
        self.print_label.setText(f"Print texto: {self.line_edit.text()}")
        self.print_label.resize(self.print_label.sizeHint())
        self.status_bar_signal.emit(f"QEdit: {self.line_edit.text()}")


class MainWindow(QMainWindow):
    # Esta señal permite comunicar la barra de estados con el resto de los widgets
    # en el formulario, incluidos el central widget.
    onchange_status_bar = pyqtSignal(str)

    def __init__(self) -> None:
        super().__init__()

        """Configuramos la geometría de la ventana."""
        self.setWindowTitle("Ventana con Boton")
        self.setGeometry(200, 100, 300, 250)

        """Configuramos las acciones."""
        ver_status = QAction(QIcon(None), "&Cambiar Status", self)
        ver_status.setStatusTip("Este es un ítem de prueba")
        ver_status.triggered.connect(self.cambiar_status_bar)

        limpiar_status = QAction(QIcon(None), "&Limpiar Status", self)
        limpiar_status.setStatusTip("Esta acción limpia la barra de estado")
        limpiar_status.triggered.connect(self.limpiar_status_bar)

        buscar = QAction(QIcon(os.path.join("img", "search_icon.png")), "&Search", self)
        buscar.setStatusTip("Un ícono de búsqueda")

        salir = QAction(QIcon(None), "&Exit", self)
        salir.setShortcut("Ctrl+Q")
        salir.setStatusTip("Salir de la aplicación")
        salir.triggered.connect(QApplication.quit)

        """Creamos la barra de menú."""
        menubar = self.menuBar()
        archivo_menu = menubar.addMenu("Archivo")  # primer menú
        archivo_menu.addAction(ver_status)
        archivo_menu.addAction(salir)

        otro_menu = menubar.addMenu("Otro Menú")  # segundo menú
        otro_menu.addAction(limpiar_status)

        """Creamos la barra de herramientas."""
        toolbar = self.addToolBar("Toolbar")
        toolbar.addAction(buscar)
        toolbar.addAction(salir)

        """Incluimos la barra de estado."""
        self.statusBar().showMessage("Listo")
        self.onchange_status_bar.connect(self.actualizar_status_bar)

        """
        Configuramos el widget central con una instancia de la clase
        MiVentana(). Además cargamos la señal en el central widget para
        que este pueda interactuar con la barra de estados de la ventana
        principal.
        """
        self.form = MiVentana(self.onchange_status_bar)
        self.setCentralWidget(self.form)

    def cambiar_status_bar(self) -> None:
        self.statusBar().showMessage("Cambié el Status")

    def limpiar_status_bar(self) -> None:
        self.statusBar().showMessage("Status limpio.")

    def actualizar_status_bar(self, msg: str) -> None:
        self.statusBar().showMessage(f"Actualizado. {msg}")
```

**En macOS**, la barra de menú no aparece dentro de la ventana, sino en la barra superior de la pantalla, junto al logo de Apple:

![](img/pyqt-mainwindow-mac.png)

### `isAutoRepeat` en `keyPressEvent`

La semana pasada vimos `keyPressEvent` para detectar cuando se presiona una tecla. Si mantenemos una tecla presionada, por defecto el método se llama **muchas veces**, como cuando mantienes una tecla presionada en un editor de texto. A veces eso es lo que queremos (por ejemplo, para mover un personaje), pero otras veces queremos reaccionar solo la primera vez (por ejemplo, para pausar el juego).

El método `isAutoRepeat()` del evento retorna `False` cuando la tecla se acaba de presionar, y `True` cuando el evento se repite por mantenerla presionada. En el siguiente ejemplo contamos cuántas veces se presiona `W` y `A`, pero solo `A` revisa `isAutoRepeat()`. Mantén ambas presionadas y compara los contadores.

Este código se encuentra en el archivo `5-recetario/2_autorepeat.py`.

```python
    def keyPressEvent(self, evento: QKeyEvent) -> None:
        if evento.key() == Qt.Key.Key_W:
            self.contador_w += 1
            self.label_w_contador.setText(f"Presionada {self.contador_w} veces")

        if evento.key() == Qt.Key.Key_A and not evento.isAutoRepeat():
            self.contador_a += 1
            self.label_a_contador.setText(f"Presionada {self.contador_a} veces")
```

### Versiones de PyQt

Todo este material usa **PyQt6**. En internet encontrarás mucho código de PyQt5, que es parecido pero no idéntico. Las diferencias más comunes son:

| PyQt5 | PyQt6 |
|---|---|
| `app.exec_()` | `app.exec()` |
| `Qt.LeftButton`, `Qt.Key_W` | `Qt.MouseButton.LeftButton`, `Qt.Key.Key_W` |
| `QAction` en `PyQt5.QtWidgets` | `QAction` en `PyQt6.QtGui` |

Si copias código de otra fuente, revisa que corresponda a PyQt6.

## Errores frecuentes

Una lista para revisar cuando algo no funciona:

| Síntoma | Causa probable | Solución |
|---|---|---|
| La ventana se congela y no responde | Un `sleep`, un `recv` o un ciclo largo en un *slot* | Usa un `QTimer` para lo periódico, o un *thread* para lo que bloquea ([*notebook* 1](1-event-loop-y-regla-de-oro.ipynb#¿Necesito-un-thread?)) |
| El programa se cae con `Segmentation fault`, o los *widgets* se comportan de forma aleatoria | Un *thread* modifica *widgets* directamente | Regla de oro: el *thread* emite señales y el *thread* principal modifica la interfaz |
| `QThread: Destroyed while thread '' is still running` y `Aborted (core dumped)` | El `QThread` solo estaba en una variable local, o se reasignó mientras corría | Guarda la referencia en un atributo o una lista |
| Un método de mi `QThread` congela la interfaz | Solo `run` se ejecuta en el *thread* nuevo. Los métodos conectados a señales se ejecutan donde vive el objeto (normalmente, el *thread* principal) | El trabajo pesado va dentro de `run` |
| Una señal se emite, pero el método conectado nunca se ejecuta | El objeto receptor se creó dentro de un `run`, en un *thread* sin *event loop* | Crea los objetos de PyQt en el *thread* principal |
| Al volver al menú y empezar otra partida, todo va al doble de velocidad | Los *threads* o *timers* de la partida anterior siguen corriendo | Detenlos al cerrar la ventana: bandera y `wait()`, o `stop()` |
| `AttributeError: 'QApplication' object has no attribute 'exec_'` | Código de PyQt5 | Usa `app.exec()` |
| El programa se cierra con `Aborted (core dumped)` después de una excepción | No hay un *hook* de excepciones | Instala `sys.excepthook` ([*notebook* 1](1-event-loop-y-regla-de-oro.ipynb#Debuggeo:-el-hook-de-excepciones)) |
| Una imagen no carga | La terminal no está en la carpeta del *script*, y el *path* relativo no se encuentra | Haz `cd` a la carpeta, o construye el *path* desde `os.path.dirname(__file__)` |
| `OSError: [Errno 98] Address already in use` al iniciar el servidor | Otro servidor sigue usando ese puerto, o el puerto sigue reservado por conexiones recién cerradas | Cierra el otro servidor. Para el segundo caso, `setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)` antes del `bind` |
| Los mensajes llegan cortados o pegados | Se usa `recv` sin un protocolo | Envía el largo primero y recibe por *chunks* ([*notebook* 4](4-gui-y-networking.ipynb#El-protocolo)) |